# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I want to answer: *which page should a content editor review first?*

The starter pipeline already proves this direction has legs: a random forest beat
a hand-tuned rule baseline by a wide margin on the same task (precision@50 went
from 0.240 to 0.740). But the starter used a proxy label — it predicted
`trend_direction == "down"` from the same window that produced the trend. What I
want to build is a future-looking label: features from one window predict an
observed outcome in the next window. That makes the problem harder and more
honest.

This lane also has the clearest decision attached to it. "Content is declining"
is a fact. "Here are the 20 pages worth your time this week, ranked" is a tool.

In [1]:
# Starter pipeline output, for reference:
#   baseline precision@50 = 0.240  →  ~12 of top 50 correct
#   random forest precision@50 = 0.740  →  ~37 of top 50 correct
# Source: outputs/model_results.json, outputs/model_report.md
#
# This confirms a learned ranking can beat a fixed rule on the starter slice.
# My lane pushes this further: a future-looking label, not a proxy derived
# from the same window.

## 2. The question: decision, action, cost of a wrong call

**Decision:** which content pages an editor should review first
for refresh, expansion, protection, or pruning — given limited weekly capacity.

**Who acts:** a content editor or content manager who triages a ranked queue
every week.

**Action:** review the top-K pages, decide whether to refresh, expand, merge,
protect, or leave alone. The system does not make the decision — it ranks
candidates so the human spends time on the most promising ones.

**Cost of a wrong call:**
- A false positive (recommended, not worth it) wastes an editor-hour on a page
  that wouldn't recover.
- A false negative (worth it, not recommended) misses a recovery window — the
  page keeps losing traffic until someone notices.

**Why ML and not a plain rule:** a plain rule like "stale + visible" catches
only 17 pages in this dataset. 9,961 pages are declining with non-trivial
demand. Picking the right 20–50 from thousands requires weighing ~10+ signals
simultaneously (volume, freshness, momentum, engagement, position, intent
type) — that's the messy, shifting pattern where ML earns its place.

In [2]:
# Editorial capacity gives this decision a real size constraint.
# A weekly triage of ~20-50 pages is realistic for one editor.
# Precision@50 is the right metric because it matches how the queue is used.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth your next 7 weeks.*

In [3]:
from pathlib import Path
import pandas as pd

for p in ["data/raw/content_refresh_anonymized.csv",
          "../../data/raw/content_refresh_anonymized.csv"]:
    if Path(p).exists():
        df = pd.read_csv(p)
        break
print(f"Total pages: {len(df):,}")
print(f"Clients:     {df['client_id'].nunique()}")
print()

# Number 1 — the pool is huge and undifferentiated
n_declining = (df["trend_direction"] == "down").sum()
print(f"1. Declining pages (any demand): {n_declining:,}  ({n_declining/len(df):.1%})")
print("   → If the answer is 'fix declining pages,' the editor gets half the catalog.\n")

# Number 2 — the meaningful pool is still large
n_declining_big = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)).sum()
print(f"2. Declining + >=500 impressions:   {n_declining_big:,}")
print("   → Still ~10k pages. A ranked queue matters.\n")

# Number 3 — content decay is real
n_page1_old = (
    (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & (df["content_age_days"] >= 180)
).sum()
print(f"3. Page 1 position + >=180 days old: {n_page1_old:,}")
print("   → ~7k pages where age + visibility suggest a refresh candidate.\n")

# Bonus — how thin is the simplest rule?
n_stale_visible = (
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
).sum()
print(f"Bonus: 'stale + visible' rule → only {n_stale_visible} pages.")
print("A plain if-statement misses most of the opportunity.")


Total pages: 30,000
Clients:     32

1. Declining pages (any demand): 16,262  (54.2%)
   → If the answer is 'fix declining pages,' the editor gets half the catalog.

2. Declining + >=500 impressions:   9,961
   → Still ~10k pages. A ranked queue matters.

3. Page 1 position + >=180 days old: 7,076
   → ~7k pages where age + visibility suggest a refresh candidate.

Bonus: 'stale + visible' rule → only 17 pages.
A plain if-statement misses most of the opportunity.


## 4. Careful words: what I can and can't claim

**What this work CAN say (decision-support, directional):**
- "Pages with these characteristics tended to decline/recover in the observed
  window — here is a ranked list of current pages that share those patterns."
- "Our ranking improved editor precision@K compared to a transparent baseline,
  under client-holdout validation."
- "Reason codes are backed by measurable thresholds, not hidden model logic."

**What this work CANNOT say (causal/absolute):**
- "Refreshing this page will cause a traffic recovery." We have no experiment,
  no control group — only observed associations.
- "This model predicts Google's algorithm." We see outcomes, not mechanisms.
- "High score = guaranteed decline." The model ranks risk; it does not predict
  the future with certainty. Every recommendation needs human review.

**The right language:** observed, associated, tended to, ranked by evidence,
  decision-support — never proved, caused, guaranteed, predicted with certainty.

In [4]:
# The target must be observed, not defined by a rule.
# My target will be an outcome measured in a future time window
# (e.g. decline/recovery over the next 30 days from the full warehouse),
# never the starter's proxy label is_declining_label.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.